In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install opencv-python mediapipe==0.10.14 protobuf==4.25.3

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input/datasets/payutch/fall-video-dataset'):
    print(dirname)

In [ ]:
%%writefile extract_keypoint.py
import os
import sys
import subprocess
import cv2
import csv
from pathlib import Path

# Khởi tạo MediaPipe Pose.
import mediapipe as mp
import mediapipe.python.solutions.pose as mp_pose

# Định nghĩa chỉ số của 17 điểm khớp xương (keypoints) cần trích xuất.
KEYPOINT_INDICES = [0, 1, 2, 3, 4, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
KEYPOINT_NAMES = [
    "Nose", "Left Eye", "Right Eye", "Left Ear", "Right Ear",
    "Left Shoulder", "Right Shoulder", "Left Elbow", "Right Elbow",
    "Left Wrist", "Right Wrist", "Left Hip", "Right Hip",
    "Left Knee", "Right Knee", "Left Ankle", "Right Ankle"
]

def extract_17_keypoints(video_path, output_csv):
    # Tạo đối tượng MediaPipe Pose.
    pose = mp_pose.Pose()
    cap = cv2.VideoCapture(video_path)
    frame_idx = 1  # Bắt đầu đếm khung hình (frame) từ 1

    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Frame", "Keypoint", "X", "Y", "Confidence"])
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(frame_rgb)

            if results.pose_landmarks:
                landmarks = results.pose_landmarks.landmark
                for idx, kp_name in zip(KEYPOINT_INDICES, KEYPOINT_NAMES):
                    lm = landmarks[idx]
                    x_pixel = lm.x * frame.shape[1]
                    y_pixel = lm.y * frame.shape[0]
                    writer.writerow([frame_idx, kp_name, x_pixel, y_pixel, lm.visibility])
            
            frame_idx += 1

    cap.release()
    pose.close()
    # print(f"Hoàn thành xử lý video, đã giải phóng bộ nhớ.") # (Tùy chọn hiển thị)

def process_dataset(input_dir, output_dir):
    # Chuyển đổi đường dẫn dạng chuỗi (string) thành đối tượng Path
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    print(f"[Thông tin] Bắt đầu xử lý dữ liệu từ thư mục: {input_dir}")
    print(f"[Thông tin] Các file CSV sẽ được lưu vào: {output_dir}")

    # Lặp qua các thư mục phân loại chính: Fall (Ngã) và No_Fall (Không ngã).
    for category in ['Fall', 'No_Fall']:
        category_input_path = input_dir / category
        print(f"[Thông tin] Đang kiểm tra thư mục: {category_input_path}")
        if not category_input_path.exists():
            print(f"[Cảnh báo] Không tìm thấy thư mục, bỏ qua: {category_input_path}")
            continue

        # Xử lý các thư mục con chứa Video gốc (Raw_Video).
        for subcat in ['Raw_Video']:
            subcat_path = category_input_path / subcat
            print(f"[Thông tin] Đang kiểm tra thư mục con: {subcat_path}")
            if not subcat_path.exists():
                print(f"[Cảnh báo] Không tìm thấy thư mục con, bỏ qua: {subcat_path}")
                continue

            # Tạo thư mục đầu ra để chứa các tệp CSV
            csv_output_dir = output_dir / category / 'Keypoints_CSV'
            print(f"[Thông tin] Đảm bảo thư mục đầu ra đã tồn tại: {csv_output_dir}")
            try:
                csv_output_dir.mkdir(parents=True, exist_ok=True)
            except OSError as e:
                print(f"[Lỗi] Không thể tạo thư mục {csv_output_dir}: {e}. Bỏ qua thư mục này.")
                continue

            # Xử lý từng file video bên trong thư mục con này.
            video_files_found = False
            print(f"[Thông tin] Đang tìm kiếm các video (.mp4, .avi, .mov) trong: {subcat_path}")
            for file_path in subcat_path.glob('*'):
                if file_path.is_file() and file_path.suffix.lower() in ['.mp4', '.avi', '.mov']:
                    video_files_found = True
                    output_csv = csv_output_dir / f"{file_path.stem}_keypoints.csv"
                    print(f"[Thông tin] Đã thấy video: {file_path}. File CSV đích: {output_csv}")

                    # Kiểm tra xem tệp CSV cho video này đã tồn tại chưa
                    if output_csv.exists():
                        print(f"[Thông tin] Bỏ qua {file_path.name}, file CSV đã có sẵn: {output_csv}")
                        continue  # Nếu có rồi thì chuyển sang video tiếp theo

                    # Tiến hành trích xuất khung xương nếu file CSV chưa bị trùng
                    print(f"[Thông tin] Đang xử lý video và xuất ra file: {output_csv}")
                    try:
                        extract_17_keypoints(str(file_path), str(output_csv))
                        # Kiểm tra xem file xuất ra có thành công và có dữ liệu không
                        if output_csv.exists():
                            if output_csv.stat().st_size > 100: # Kiểm tra dung lượng > header
                                print(f"[Thành công] Đã lưu keypoints tại: {output_csv} (Kích thước: {output_csv.stat().st_size} bytes)")
                            else:
                                print(f"[Cảnh báo] File CSV đã tạo nhưng quá nhỏ (có thể rỗng hoặc chỉ chứa tiêu đề): {output_csv} (Kích thước: {output_csv.stat().st_size} bytes)")
                        else:
                             print(f"[Lỗi] Tệp CSV đầu ra chưa được tạo ra: {output_csv}")
                    except Exception as e:
                        print(f"[Lỗi] Ngoại lệ xảy ra trong lúc trích xuất keypoints cho video {file_path}: {e}")

            if not video_files_found:
                 print(f"[Cảnh báo] Không tìm thấy tệp video nào (.mp4, .avi, .mov) tại {subcat_path}")

if __name__ == "__main__":
    # Cấu hình các đường dẫn Đầu vào/Đầu ra trên Kaggle
    INPUT_DIR = '/kaggle/input/datasets/payutch/fall-video-dataset'
    OUTPUT_DIR = '/kaggle/working/Fall-Detection/csv_data'

    print(f"[Cài đặt] Thư mục gốc chứa Dataset (Đầu vào): {INPUT_DIR}")
    print(f"[Cài đặt] Thư mục lưu file CSV (Đầu ra): {OUTPUT_DIR}")

    input_path = Path(INPUT_DIR)
    
    if not input_path.exists() or not input_path.is_dir():
        print(f"[Lỗi] Không tìm thấy thư mục gốc chứa Dataset ở đường dẫn: {input_path}")
        print("Vui lòng kiểm tra xem bạn đã copy/upload các video thô (raw videos) vào đúng đường dẫn chưa.")
    else:
        print(f"[Thông tin] Đã tìm thấy thư mục đầu vào. Bắt đầu quá trình xử lý...")
        try:
            process_dataset(INPUT_DIR, OUTPUT_DIR)
        except Exception as e:
            print(f"[Lỗi nghiêm trọng] Xảy ra lỗi ngoài dự kiến trong khi xử lý dữ liệu: {e}")
    print("[Thông tin] Quá trình chạy kịch bản (Script) đã hoàn tất.")


In [1]:
%%writefile common_keypoints.py
NUM_KEYPOINTS = 17
KEYPOINT_ORDER_ALPHABETICAL = [
    "Left Ankle", "Left Ear", "Left Elbow", "Left Eye", "Left Hip",
    "Left Knee", "Left Shoulder", "Left Wrist", "Nose", "Right Ankle",
    "Right Ear", "Right Elbow", "Right Eye", "Right Hip", "Right Knee",
    "Right Shoulder", "Right Wrist"
]

Writing common_keypoints.py


In [2]:
%%writefile create_seq.py
# --- START OF FILE create_seq.py ---
import os
import numpy as np
import pandas as pd
from pathlib import Path
import logging
import time
import json
import argparse

# Import from common_keypoints
try:
    from common_keypoints import KEYPOINT_ORDER_ALPHABETICAL, NUM_KEYPOINTS as COMMON_NUM_KEYPOINTS
except ImportError:
    print("LỖI NGHIÊM TRỌNG: Không tìm thấy file common_keypoints.py. Kịch bản này không thể chạy nếu thiếu nó.")
    exit()

# --- Configuration ---
INPUT_ROOT_DIR = Path("/kaggle/input/notebooks/nhathoanghuynh/notebookc03ddf911b/Fall-Detection/csv_data")
OUTPUT_ROOT_DIR = Path("/kaggle/working/Fall-Detection/numpy_data")
SEQUENCE_LENGTH = 60
STEP_SIZE_FALL = 12
STEP_SIZE_NO_FALL = 30
LABELS = {"Fall": 1, "No_Fall": 0}

NUM_KEYPOINTS = COMMON_NUM_KEYPOINTS
NUM_FEATURES_PER_KEYPOINT = 3
EXPECTED_FEATURES_PER_FRAME = NUM_KEYPOINTS * NUM_FEATURES_PER_KEYPOINT

# --- Logging Setup (ย้ายมาไว้ก่อนเพื่อให้ KEYPOINT logging แสดงผล) ---
OUTPUT_ROOT_DIR.mkdir(parents=True, exist_ok=True)
temp_log_filename_for_type_detection = "temp_arg_parse_log.txt"
parser_temp = argparse.ArgumentParser(add_help=False)
parser_temp.add_argument("--process_type", type=str, default="both", choices=["fall", "no_fall", "both"])
args_temp, _ = parser_temp.parse_known_args()

log_file_name = f"create_seq_{args_temp.process_type}_{time.strftime('%Y%m%d-%H%M%S')}.log"
log_file_path = OUTPUT_ROOT_DIR / log_file_name

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s",
    handlers=[logging.FileHandler(log_file_path, mode='w'), logging.StreamHandler()]
)

logging.info(f"--- Đang khởi tạo thứ tự Keypoint cho file .npy đầu ra ---")
logging.info(f"Sử dụng KEYPOINT_ORDER_ALPHABETICAL cho cấu trúc .npy: {KEYPOINT_ORDER_ALPHABETICAL}")
if NUM_KEYPOINTS != 17:
    logging.error(f"LỖI NGHIÊM TRỌNG: NUM_KEYPOINTS từ common_keypoints.py là {NUM_KEYPOINTS}, kỳ vọng là 17.")
    exit()
if EXPECTED_FEATURES_PER_FRAME != 51:
    logging.error(f"LỖI NGHIÊM TRỌNG: EXPECTED_FEATURES_PER_FRAME là {EXPECTED_FEATURES_PER_FRAME}, kỳ vọng là 51.")
    exit()
logging.info("---------------------------------------------------")

# --- Helper Functions ---
def process_csv_for_sequencing(csv_path: Path, sequence_length: int, step_size: int, category_name: str):
    sequences = []
    sequence_file_info = []
    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            logging.warning(f"File CSV trống: {csv_path}. Đang bỏ qua.")
            return [], []

        required_cols = ['Frame', 'Keypoint', 'X', 'Y', 'Confidence']
        if not all(col in df.columns for col in required_cols):
            logging.error(f"Thiếu một hoặc nhiều cột thiết yếu trong {csv_path}. Bỏ qua file này.")
            return [], []

        for col_to_check in ['Frame', 'X', 'Y', 'Confidence']:
            if not pd.api.types.is_numeric_dtype(df[col_to_check]):
                try:
                    df[col_to_check] = pd.to_numeric(df[col_to_check], errors='coerce')
                    if df[col_to_check].isnull().any():
                         logging.warning(f"Xuất hiện giá trị NaN trong cột '{col_to_check}' khi chuyển đổi số liệu cho {csv_path}.")
                except ValueError:
                    logging.error(f"Không thể chuyển đổi '{col_to_check}' sang số trong {csv_path}. Bỏ qua file này.")
                    return [],[]
        
        if df['Keypoint'].nunique() == 0:
            logging.warning(f"Không tìm thấy cột keypoint khả dụng nào để xoay (pivot) trong {csv_path}. Bỏ qua file này.")
            return [], []

        frame_features = df.pivot_table(
            index='Frame', columns='Keypoint', values=['X', 'Y', 'Confidence']
        )
        frame_features.columns = ['_'.join(map(str, col)) for col in frame_features.columns.values]

        if df['Frame'].empty or df['Frame'].min() > df['Frame'].max():
            logging.warning(f"Không tìm thấy khung hình hợp lệ trong {csv_path}. Đang bỏ qua.")
            return [], []

        min_frame = int(df['Frame'].min())
        max_frame = int(df['Frame'].max())
        all_frames_index = pd.RangeIndex(start=min_frame, stop=max_frame + 1, name='Frame')

        # --- Define expected columns in ALPHABETICAL order using imported KEYPOINT_ORDER_ALPHABETICAL ---
        expected_cols_alphabetical_ordered = []
        for kp_name in KEYPOINT_ORDER_ALPHABETICAL: # Use imported ALPHABETICAL order
            expected_cols_alphabetical_ordered.append(f"X_{kp_name}")
            expected_cols_alphabetical_ordered.append(f"Y_{kp_name}")
            expected_cols_alphabetical_ordered.append(f"Confidence_{kp_name}")
        
        # Reindex with all frames and ensure all expected columns are present and in ALPHABETICAL order
        frame_features = frame_features.reindex(index=all_frames_index, columns=expected_cols_alphabetical_ordered)

        # --- Interpolation for X and Y (iterating based on ALPHABETICAL order) ---
        for kp_name in KEYPOINT_ORDER_ALPHABETICAL:
            x_col = f"X_{kp_name}"
            y_col = f"Y_{kp_name}"
            if x_col in frame_features.columns:
                frame_features[x_col] = frame_features[x_col].interpolate(method='linear', limit_direction='both', limit_area='inside')
            if y_col in frame_features.columns:
                frame_features[y_col] = frame_features[y_col].interpolate(method='linear', limit_direction='both', limit_area='inside')
        
        frame_features = frame_features.fillna(0.0)

        if frame_features.shape[1] != EXPECTED_FEATURES_PER_FRAME:
             logging.critical(f"LỖI NGHIÊM TRỌNG: Lệch số lượng feature của {csv_path} SAU KHI xử lý. Kỳ vọng {EXPECTED_FEATURES_PER_FRAME}, nhưng nhận được {frame_features.shape[1]}. Các cột hiện tại: {frame_features.columns.tolist()}")
             return [], []

        data = frame_features.values
        num_frames = data.shape[0]

        if num_frames == 0:
            logging.warning(f"Không còn dòng dữ liệu nào sau khi xử lý đối với {csv_path}. Đang bỏ qua.")
            return [], []

        base_name_for_file = csv_path.stem.replace("_keypoints", "")

        if num_frames < sequence_length:
            if num_frames > 0:
                padding_needed = sequence_length - num_frames
                last_frame_data = data[-1, :].reshape(1, -1)
                padding_data = np.repeat(last_frame_data, padding_needed, axis=0)
                padded_sequence = np.vstack((data, padding_data))
                sequences.append(padded_sequence)
                sequence_file_info.append((base_name_for_file, csv_path.name, 0, category_name.lower()))
                logging.debug(f"Đã đệm file {csv_path.name}: từ {num_frames} lên {sequence_length} khung hình.")
            else:
                logging.warning(f"Bỏ qua quá trình đệm cho {csv_path.name} vì nó có 0 khung hình hợp lệ.")
        else:
            seq_idx = 0
            for i in range(0, num_frames - sequence_length + 1, step_size):
                seq = data[i : i + sequence_length]
                sequences.append(seq)
                sequence_file_info.append((base_name_for_file, csv_path.name, seq_idx, category_name.lower()))
                seq_idx += 1
            if seq_idx == 0:
                logging.info(f"Không thể cắt ra chuỗi đầy đủ từ sliding window cho {csv_path.name} (Số khung hình: {num_frames}, Độ dài chuỗi: {sequence_length}, Bước nhảy: {step_size})")
        return sequences, sequence_file_info
    except pd.errors.EmptyDataError:
        logging.warning(f"File CSV {csv_path} trống hoặc không hợp lệ. Đang bỏ qua.")
        return [], []
    except Exception as e:
        logging.error(f"Xử lý CSV thất bại {csv_path}: {e}", exc_info=True)
        return [], []

# --- Main Processing Logic ---
def main():
    parser = argparse.ArgumentParser(description="Process CSV keypoint data into ALPHABETICALLY ordered sequences with padding and interpolation.")
    parser.add_argument(
        "--process_type", type=str, choices=["fall", "no_fall", "both"], default="both",
        help="Kiểu dữ liệu cần xử lý: 'fall', 'no_fall', hoặc 'both'. Mặc định là 'both'."
    )
    args = parser.parse_args() # Parsed after logging is setup, but this is fine.

    logging.info(f"--- Cấu hình hệ thống ---")
    logging.info(f"Loại dữ liệu cần trích: {args.process_type}")
    logging.info(f"Thư mục CSV gốc (Đầu vào): {INPUT_ROOT_DIR.resolve()}")
    logging.info(f"Thư mục NPY (Đầu ra): {OUTPUT_ROOT_DIR.resolve()} (Keypoint trong .npy sẽ được sắp xếp theo thứ tự ABC)")
    logging.info(f"Độ dài sequence (Sequence Length): {SEQUENCE_LENGTH} frames")
    logging.info(f"Khoảng nhảy Step Size cho thư mục (Fall): {STEP_SIZE_FALL}")
    logging.info(f"Khoảng nhảy Step Size cho thư mục (No_Fall): {STEP_SIZE_NO_FALL}")
    logging.info("Đệm thêm vào video ngắn bằng khung hình cuối. Nội suy giả lập những trục X,Y bị thiếu.")
    logging.info("--- Hết phần cấu hình ---")

    categories_to_process = {}
    if args.process_type == "fall":
        categories_to_process = {"Fall": LABELS["Fall"]}
    elif args.process_type == "no_fall":
        categories_to_process = {"No_Fall": LABELS["No_Fall"]}
    elif args.process_type == "both":
        categories_to_process = LABELS
    else:
        logging.error(f"Invalid process_type: {args.process_type}")
        return

    logging.info("Đang tính toán sơ bộ để dự đoán số sequences dự kiến ​​của từng phân loại data...")
    expected_sequence_counts = {}
    all_potential_file_actions = []

    for category, label in categories_to_process.items():
        category_input_path = INPUT_ROOT_DIR / category / "Keypoints_CSV"
        expected_sequence_counts[category] = 0
        if not category_input_path.is_dir():
            logging.warning(f"Tính toán sơ bộ: Không tìm thấy Thư mục này: {category_input_path}")
            continue
        csv_files = list(category_input_path.glob("*.csv"))
        if not csv_files:
            logging.warning(f"Tính toán sơ bộ: Không tìm ra File CSV nào cả trong: {category_input_path}")
            continue
        
        current_category_file_infos = []
        for csv_file in csv_files:
            try:
                df_temp = pd.read_csv(csv_file, usecols=['Frame'])
                if df_temp.empty or 'Frame' not in df_temp.columns or df_temp['Frame'].nunique() == 0 :
                    logging.warning(f"Nạp trước thông số tính toán: Bỏ qua lỗi trống rỗng hoặc hỏng frame data ở {csv_file.name}")
                    continue
                num_unique_frames = df_temp['Frame'].nunique()
                current_category_file_infos.append((csv_file, category, label, num_unique_frames))
                current_step_size = STEP_SIZE_FALL if category == "Fall" else STEP_SIZE_NO_FALL
                if num_unique_frames < SEQUENCE_LENGTH:
                    if num_unique_frames > 0: 
                        expected_sequence_counts[category] += 1
                else:
                    if num_unique_frames >= SEQUENCE_LENGTH:
                        num_possible_seqs = (num_unique_frames - SEQUENCE_LENGTH) // current_step_size + 1
                        expected_sequence_counts[category] += num_possible_seqs
            except Exception as e:
                logging.warning(f"Nháp số liệu: Lỗi nhẹ khi đọc qua đếm frame file {csv_file}: {e}")
        all_potential_file_actions.extend(current_category_file_infos)
    
    logging.info("--- Số lượng Sequences xấp xỉ sẽ tạo ra (Thông số tính thử) ---")
    total_expected_sequences = 0
    for category, count in expected_sequence_counts.items():
        logging.info(f"Thể loại/Phân loại '{category}': Sinh ra {count} mảnh sequence array")
        total_expected_sequences += count
    logging.info(f"Tổng cộng số lượng arrays dự kiến cần chuẩn bị: {total_expected_sequences}")
    logging.info("---------------------------------------------")

    if total_expected_sequences == 0:
        logging.warning("Sẽ không có bảng Array (.NPY) dữ liệu khung xương nào được xuất ra cả. Kết thúc Script.")
        return

    proceed = "y"
    
    logging.info("Tiếp tục bắt đầu tạo file sequence .npy ...")
    all_sequences_info_list_for_json = []
    base_temp_npy_dir = OUTPUT_ROOT_DIR / "_temp_npy_sequences" # Generic name for temp dir
    base_temp_npy_dir.mkdir(parents=True, exist_ok=True)

    processed_csv_files_count = 0
    saved_sequences_count = 0

    for csv_file_path, category_name, label, _ in all_potential_file_actions:
        category_output_npy_dir = base_temp_npy_dir / category_name
        category_output_npy_dir.mkdir(parents=True, exist_ok=True)
        current_step_size_for_file = STEP_SIZE_FALL if category_name == "Fall" else STEP_SIZE_NO_FALL
        logging.info(f"Đang tiến hành xử lý chuyển đổi: {csv_file_path.name} [Phân loại nhánh: {category_name}]")
        # Removed expected_features argument as it's now a global constant based on NUM_KEYPOINTS
        video_sequences, generated_file_infos = process_csv_for_sequencing(
            csv_file_path, SEQUENCE_LENGTH, current_step_size_for_file, category_name
        )

        if not video_sequences:
            logging.warning(f"Cảnh báo: Không thể sinh ra mảng chunk nào cho file trống {csv_file_path.name}")
            continue
        processed_csv_files_count +=1

        for seq_data, (base_name, _, seq_idx, cat_name_lower) in zip(video_sequences, generated_file_infos):
            if seq_data.shape != (SEQUENCE_LENGTH, EXPECTED_FEATURES_PER_FRAME):
                 logging.error(f"Khúc array được cắt {seq_idx} trích từ của video {base_name} trong file CSV {csv_file_path.name} BỊ SAI KÍCH THƯỚC CHIỀU (Shape): {seq_data.shape}. Nó phải là đúng y sì thế này: {(SEQUENCE_LENGTH, EXPECTED_FEATURES_PER_FRAME)}. Đang bỏ qua việc lưu.")
                 continue
            npy_filename = f"{base_name}_{cat_name_lower}_seq_{seq_idx:03d}.npy"
            npy_path = category_output_npy_dir / npy_filename
            try:
                np.save(npy_path, seq_data.astype(np.float32))
                logging.debug(f"Đã lưu thành công mảng vào: {npy_path} (Khung Shape: {seq_data.shape}) - Keypoints xếp chuẩn xác theo Alphabet(ABC).")
                all_sequences_info_list_for_json.append((str(npy_path), label))
                saved_sequences_count +=1
            except Exception as e:
                logging.error(f"Save thất bại không thể ghi ra file {npy_path}: {e}", exc_info=True)

    logging.info(f"Hoạt động xử lý chạy hoàn tất. Đã băm cấu trúc được chuỗi data xịn trên {processed_csv_files_count} file CSV.")
    logging.info(f"Toàn bộ tổng cục các Sequences mảng khối (.NPY) đã lưu: {saved_sequences_count} chuỗi Array")
    
    sequences_info_file_name = f"sequences_info_padded_interpolated_{args.process_type}_{time.strftime('%Y%m%d-%H%M%S')}.json"
    sequences_info_file_path = OUTPUT_ROOT_DIR / sequences_info_file_name
    try:
        with open(sequences_info_file_path, 'w') as f:
            json.dump(all_sequences_info_list_for_json, f, indent=4)
        logging.info(f"Thành công ghi nhớ danh sách map metadata json của toàn bộ Sequence vào địa chỉ: {sequences_info_file_path}")
    except Exception as e:
        logging.error(f"Cảnh báo lỗi: Không ghi tạo được json information meta tại {sequences_info_file_path}: {e}")

    logging.info(f"Hoàn thành toàn bộ quy định Trích xuất CSV thành Sequence thành công! Kết thúc tiến trình npy sinh mảng.")

if __name__ == "__main__":
    main()
# --- END OF FILE create_seq.py ---

Writing create_seq.py


In [3]:
!python create_seq.py

2026-04-11 09:42:27,637 [INFO] - --- Đang khởi tạo thứ tự Keypoint cho file .npy đầu ra ---
2026-04-11 09:42:27,637 [INFO] - Sử dụng KEYPOINT_ORDER_ALPHABETICAL cho cấu trúc .npy: ['Left Ankle', 'Left Ear', 'Left Elbow', 'Left Eye', 'Left Hip', 'Left Knee', 'Left Shoulder', 'Left Wrist', 'Nose', 'Right Ankle', 'Right Ear', 'Right Elbow', 'Right Eye', 'Right Hip', 'Right Knee', 'Right Shoulder', 'Right Wrist']
2026-04-11 09:42:27,637 [INFO] - ---------------------------------------------------
2026-04-11 09:42:27,637 [INFO] - --- Cấu hình hệ thống ---
2026-04-11 09:42:27,637 [INFO] - Loại dữ liệu cần trích: both
2026-04-11 09:42:27,644 [INFO] - Thư mục CSV gốc (Đầu vào): /kaggle/input/notebooks/nhathoanghuynh/notebookc03ddf911b/Fall-Detection/csv_data
2026-04-11 09:42:27,644 [INFO] - Thư mục NPY (Đầu ra): /kaggle/working/Fall-Detection/numpy_data (Keypoint trong .npy sẽ được sắp xếp theo thứ tự ABC)
2026-04-11 09:42:27,645 [INFO] - Độ dài sequence (Sequence Length): 60 frames
2026-04-11

In [5]:
%%writefile split_dataset_v2.py
# --- START OF FILE split_dataset.py ---

import os
import numpy as np
import pandas as pd
from pathlib import Path
import logging
import shutil
import json
from sklearn.model_selection import train_test_split
import argparse
import time 

# --- Configuration ---
PROCESSED_DATA_ROOT = Path("/kaggle/working/Fall-Detection/numpy_data")
FINAL_SPLIT_OUTPUT_ROOT = Path("/kaggle/working/Fall-Detection/Final_Dataset_Splits")
SPLIT_RATIOS = {"train": 0.7, "val": 0.15, "test": 0.15}
TEMP_NPY_DIR_NAME = "_temp_npy_sequences"

# --- Helper Functions (if any, or can be inline) ---

def find_latest_sequences_info_file(processed_data_root: Path, process_type: str) -> Path | None:
    """Finds the latest sequence info JSON file based on process_type."""
    pattern = f"sequences_info_padded_interpolated_{process_type}_*.json"
    files = sorted(processed_data_root.glob(pattern), key=os.path.getmtime, reverse=True)
    if files:
        logging.info(f"Đã tìm thấy file thông tin sequence: {files[0]}")
        return files[0]
    else:
        generic_pattern = "sequences_info_padded_interpolated_both_*.json"
        files = sorted(processed_data_root.glob(generic_pattern), key=os.path.getmtime, reverse=True)
        if files:
            logging.warning(f"Không tìm thấy phân loại {process_type}, đang dùng file chung mới nhất: {files[0]}")
            return files[0]
        logging.error(f"Không có file thông tin json nào theo cấu trúc '{pattern}' hoặc 'both' tại {processed_data_root}.")
        return None

# --- Main Processing Logic ---
def main():
    parser = argparse.ArgumentParser(description="Chia nhỏ dataset thành train, validation, test.")
    parser.add_argument(
        "--process_type_for_json", type=str, default="both", choices=["fall", "no_fall", "both"],
        help="Cái này để chỉ định file json của process_type nào cần dùng."
    )
    args = parser.parse_args()

    FINAL_SPLIT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    log_file_path = FINAL_SPLIT_OUTPUT_ROOT / f"split_dataset_{time.strftime('%Y%m%d-%H%M%S')}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] - %(message)s",
        handlers=[
            logging.FileHandler(log_file_path, mode='w'),
            logging.StreamHandler()
        ]
    )

    logging.info("Bắt đầu quá trình Băm Split Data và chia nhóm tập tin...")
    logging.info(f"Nguồn Data .npy và JSON đầu vào nằm ở: {PROCESSED_DATA_ROOT.resolve()}")
    logging.info(f"Nguồn Output thư mục Data cuối cùng nằm tại: {FINAL_SPLIT_OUTPUT_ROOT.resolve()}")

    sequences_info_file_path = find_latest_sequences_info_file(PROCESSED_DATA_ROOT, args.process_type_for_json)
    if not sequences_info_file_path or not sequences_info_file_path.exists():
        logging.error(f"Khóa lỗi tìm kiếm Json. Mong ngài hãy chạy create_seq.py trước và trỏ đúng đường dẫn.")
        return

    logging.info(f"Đang nạp file sequence json: {sequences_info_file_path}...")
    try:
        with open(sequences_info_file_path, 'r') as f:
            all_sequences_info_loaded = json.load(f)
    except Exception as e:
        logging.error(f"File json đọc lỗi rồi {sequences_info_file_path}: {e}")
        return

    if not all_sequences_info_loaded:
        logging.error("File json đọc ra rỗng không có dòng nào. Thoát tiến trình.")
        return

    all_sequences_info = [(Path(item[0]), item[1]) for item in all_sequences_info_loaded]
    logging.info(f"Đã đọc và tải file thành công, nạp được {len(all_sequences_info)} mảnh sequences vào RAM.")

    logging.info("Chuẩn bị tỉ lệ chia nhỏ Data thành train, validation, test (Máy học)...")
    sequence_paths_from_json = [info[0] for info in all_sequences_info]
    sequence_labels = [info[1] for info in all_sequences_info]

    if len(set(sequence_labels)) < 2 and len(sequence_labels) > 0:
        logging.warning("Chỉ thấy 1 Class trong mảng. Tiến trình băm tỉ lệ kiểu Stratified có thể sẽ thất bại do Class Balance.")
    elif not sequence_labels:
         logging.error("Không có các mảnh nhãn label để băm tỷ lệ. Tắt quá trình.")
         return

    try:
        train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
            sequence_paths_from_json, sequence_labels,
            test_size=SPLIT_RATIOS["test"],
            random_state=42,
            stratify=sequence_labels if len(set(sequence_labels)) > 1 else None
        )

        if SPLIT_RATIOS["train"] + SPLIT_RATIOS["val"] == 0:
            val_relative_size = 0
        else:
            val_relative_size = SPLIT_RATIOS["val"] / (SPLIT_RATIOS["train"] + SPLIT_RATIOS["val"])

        if train_val_paths:
             can_stratify_second_split = len(set(train_val_labels)) > 1 and len(train_val_paths) > 1
             if not can_stratify_second_split and len(train_val_paths) > 1:
                  logging.warning("Chỉ có 1 Lớp Label duy nhất để mổ Val. Lược bỏ Stratified.")
            
             train_paths, val_paths, train_labels, val_labels = train_test_split(
                 train_val_paths, train_val_labels,
                 test_size=val_relative_size if val_relative_size > 0 and len(train_val_paths) * val_relative_size >=1 else 0, # Ensure test_size is valid
                 random_state=42,
                 stratify=train_val_labels if can_stratify_second_split else None
             )
        else:
             train_paths, val_paths, train_labels, val_labels = [], [], [], []
             logging.warning("Kho Train+Validation sau khi cắt xong có nội dung rỗng.")

        split_data = {
            "train": list(zip(train_paths, train_labels)),
            "val": list(zip(val_paths, val_labels)),
            "test": list(zip(test_paths, test_labels))
        }
        logging.info(f"Đã cắt tỉ lệ phân vùng thành công. Độ lớn: Train={len(train_paths)}, Val={len(val_paths)}, Test={len(test_paths)}")

    except ValueError as e:
         logging.error(f"Xảy ra lỗi Cắt TrainTest. Thường là do Quá ít tập dữ liệu có trong Dataset. Thông báo lỗi gốc do ScikitLearn gửi ra: {e}")
         logging.warning("Dự phòng Fallback: Toàn bộ sẽ tự động đẩy sang thư mục 'train'.")
         split_data = {
            "train": list(zip(sequence_paths_from_json, sequence_labels)),
            "val": [],
            "test": []
         }

    logging.info("Đang luân chuyển và tổ chức lại File Array vào các thư mục mục tiêu có kiến trúc xịn. Đồng thời kết xuất Summary metadata CSV...")
    summary_data = []
    
    for split_name, sequences_in_split in split_data.items():
        logging.info(f"Chuẩn bị setup cho thư mục Set: {split_name}...")
        if not sequences_in_split:
            logging.info(f"Rỗng không có mảng Array nào để xử lý cho set {split_name}. Tự động nhảy qua phần khác.")
            continue
        for source_npy_path, label in sequences_in_split:
            target_label_dir_name = "fall" if label == 1 else "no_fall"
            target_split_label_dir = FINAL_SPLIT_OUTPUT_ROOT / split_name / target_label_dir_name
            target_split_label_dir.mkdir(parents=True, exist_ok=True)
            target_path = target_split_label_dir / source_npy_path.name
            
            try:
                if not source_npy_path.exists():
                    logging.error(f"Source file {source_npy_path} (nắm trong cái file JSON Map) nó không hề có thật. Quái lạ, bỏ qua vậy.")
                    continue
                shutil.copy2(str(source_npy_path), str(target_path))
                logging.debug(f"Đã copy xong {source_npy_path.name} đến {target_path}")
                summary_data.append({
                    "filename": target_path.name,
                    "label": label,
                    "split": split_name,
                    "final_path_relative": str(target_path.relative_to(FINAL_SPLIT_OUTPUT_ROOT))
                })
            except FileNotFoundError:
                 logging.error(f"File mảng Array bị thiếu {source_npy_path}. Bỏ.")
            except Exception as e:
                logging.error(f"Copy {source_npy_path.name} tới {target_path} bị lỗi: {e}")

    temp_npy_sequences_dir = PROCESSED_DATA_ROOT / TEMP_NPY_DIR_NAME
    logging.info(f"Kiểm toán Directory Temp NPY (Nên Xóa tay): {temp_npy_sequences_dir}")
    if temp_npy_sequences_dir.exists() and temp_npy_sequences_dir.is_dir():
        logging.info(f"Folder tạm _temp_npy_sequences đang còn tồn tại: {temp_npy_sequences_dir}. "
                     f"Sau khi bạn check hàng tại {FINAL_SPLIT_OUTPUT_ROOT} đầy đủ hãy tự xóa cái rác folder tạm đó nha.")
    else:
        logging.info(f"Khu rác _temp_npy_sequences cho {temp_npy_sequences_dir} đã tàng hình. Vậy là tốt.")

    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        summary_csv_path = FINAL_SPLIT_OUTPUT_ROOT / "dataset_summary.csv"
        try:
            summary_df.to_csv(summary_csv_path, index=False)
            logging.info(f"Thành công ghi nhớ danh sách map metadata Dataset CSV của toàn bộ thư mục tới địa chỉ chốt: {summary_csv_path}")
        except Exception as e:
            logging.error(f"Cảnh báo lỗi: Không ghi tạo được CSV summary csv {e}")
    else:
        logging.warning("Không có bất kì file data Array nào được tống sang cả. Không sinh ra summary.csv.")

    logging.info("Tiến trình băm thư mục Array ra các tập Validation Test... Cực kỳ thành công!")

if __name__ == "__main__":
    main()
# --- END OF FILE split_dataset.py ---

Overwriting split_dataset_v2.py


In [6]:
!python split_dataset_v2.py

2026-04-11 09:53:05,463 [INFO] - Bắt đầu quá trình Băm Split Data và chia nhóm tập tin...
2026-04-11 09:53:05,464 [INFO] - Nguồn Data .npy và JSON đầu vào nằm ở: /kaggle/working/Fall-Detection/numpy_data
2026-04-11 09:53:05,464 [INFO] - Nguồn Output thư mục Data cuối cùng nằm tại: /kaggle/working/Fall-Detection/Final_Dataset_Splits
2026-04-11 09:53:05,464 [INFO] - Đã tìm thấy file thông tin sequence: /kaggle/working/Fall-Detection/numpy_data/sequences_info_padded_interpolated_both_20260411-094614.json
2026-04-11 09:53:05,464 [INFO] - Đang nạp file sequence json: /kaggle/working/Fall-Detection/numpy_data/sequences_info_padded_interpolated_both_20260411-094614.json...
2026-04-11 09:53:05,498 [INFO] - Đã đọc và tải file thành công, nạp được 16441 mảnh sequences vào RAM.
2026-04-11 09:53:05,498 [INFO] - Chuẩn bị tỉ lệ chia nhỏ Data thành train, validation, test (Máy học)...
2026-04-11 09:53:05,521 [INFO] - Đã cắt tỉ lệ phân vùng thành công. Độ lớn: Train=11508, Val=2466, Test=2467
2026-04-